## Module A — Data preprocessing & merge (**daily**, corrected)\n
\n
This notebook replaces the earlier (incorrect) annual-mean pipeline. It:\n
\n
- Cleans **NFLX** and **GLD** daily data\n
- Aligns the shared trading calendar via an **inner join on `Date`**\n
- Quantifies coverage / missingness\n
- Exports daily aligned **prices** and **log returns** for downstream modules\n
\n
Outputs:\n
- `outputs/combined_daily_prices.csv`\n
- `outputs/combined_daily_logreturns.csv`\n

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
# VS Code sometimes sets the notebook working directory to `notebooks/`.
# If we don't see the data files here, step up one level.
if not (ROOT / "NFLX.csv").exists():
    ROOT = ROOT.parent

OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

nflx_path = ROOT / "NFLX.csv"
gld_path = ROOT / "navhist-us-en-gld(navhist).csv"

nflx_path, gld_path


(WindowsPath('c:/Users/ASUS/Desktop/foundation of analysis/final_project_0/notebooks/NFLX.csv'),
 WindowsPath('c:/Users/ASUS/Desktop/foundation of analysis/final_project_0/notebooks/navhist-us-en-gld(navhist).csv'))

In [3]:
# --- Load & clean NFLX ---
nflx = pd.read_csv(nflx_path)
nflx["Date"] = pd.to_datetime(nflx["Date"], errors="coerce")
nflx["Adj Close"] = pd.to_numeric(nflx["Adj Close"], errors="coerce")
nflx = nflx.dropna(subset=["Date", "Adj Close"]).sort_values("Date")
nflx = nflx[["Date", "Adj Close"]].rename(columns={"Adj Close": "NFLX_AdjClose"})

nflx.head()


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\ASUS\\Desktop\\foundation of analysis\\final_project_0\\notebooks\\NFLX.csv'

In [ ]:
# --- Load & clean GLD (metadata rows + encoding fallback) ---
last_err = None
for enc in ("utf-8-sig", "cp1252", "latin1"):
    try:
        gld = pd.read_csv(gld_path, skiprows=3, encoding=enc)
        last_err = None
        break
    except Exception as e:
        last_err = e
if last_err is not None:
    raise last_err

gld["Date"] = pd.to_datetime(gld["Date"], format="%d-%b-%Y", errors="coerce")
gld["NAV"] = pd.to_numeric(gld["NAV"], errors="coerce")
gld = gld.dropna(subset=["Date", "NAV"]).sort_values("Date")
gld = gld[["Date", "NAV"]].rename(columns={"NAV": "GLD_NAV"})

gld.head()


In [ ]:
# --- Align trading days and quantify coverage ---
merged_prices = (
    pd.merge(nflx, gld, on="Date", how="inner")
    .sort_values("Date")
    .reset_index(drop=True)
)

coverage = {
    "nflx_days": int(len(nflx)),
    "gld_days": int(len(gld)),
    "merged_days": int(len(merged_prices)),
    "min_date": str(merged_prices["Date"].min().date()),
    "max_date": str(merged_prices["Date"].max().date()),
}
coverage


In [ ]:
# --- Compute daily log returns ---
merged = merged_prices.copy()
merged["NFLX_logret"] = np.log(merged["NFLX_AdjClose"]).diff()
merged["GLD_logret"] = np.log(merged["GLD_NAV"]).diff()
merged = merged.dropna(subset=["NFLX_logret", "GLD_logret"]).reset_index(drop=True)

merged.head()


In [ ]:
# --- Export for downstream modules ---
prices_out = OUT_DIR / "combined_daily_prices.csv"
rets_out = OUT_DIR / "combined_daily_logreturns.csv"

merged_prices.to_csv(prices_out, index=False)
merged.to_csv(rets_out, index=False)

prices_out, rets_out
